# Évaluation finale sur le jeu de test
## Reconnaissance du genre - noms propres ivoiriens | SIGAP / AFOR
**INP-HB · ESI · Stage 2A · 2025-2026**

Évaluation **unique** sur le test des modèles retenus sur la validation :

| Modèle | Origine |
|---|---|
| Régression logistique réglée | `optimisation.ipynb` (n-grammes 1-6 + mots, C = 2) |
| v5 (modèle par mot), 3 graines | meilleure configuration Optuna, `optimisation.ipynb` |
| v4 (CNN + BiLSTM + MHA), 3 graines | meilleure augmentation des ablations, `optimisation.ipynb` |
| Meilleur gradient boosting | meilleur sur la validation, `boosting.ipynb` |
| Ensembles avec la régression logistique | moyenne des probabilités |

**Règles fixées avant de regarder le test**
- Modèle de production : la régression logistique (le plus simple), **sauf** si un autre modèle
  a une meilleure accuracy **et** une différence significative au test de McNemar (p < 0,05).
- Mode « incertain » : seuil de confiance **0,70**, choisi sur la validation.

## 0. Paramètres

In [ ]:
FINAL_DIR       = 'runs/final'
FIGURES_DIR     = 'figures'
SEUIL_INCERTAIN = 0.70
N_BOOT          = 2000

## 1. Données, découpage figé et régression logistique

In [ ]:
import os, re, glob, json, hashlib, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union
warnings.filterwarnings('ignore', category=UserWarning)
os.makedirs(FIGURES_DIR, exist_ok=True)


def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    text = re.sub(r"['-]", ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()


df = pd.read_csv('dataset_nettoye.csv')
df['full_name'] = (df['NOM'].apply(clean_text) + ' ' + df['PRENOMS'].fillna('').apply(clean_text)).str.strip()
df = df[df['full_name'] != ''].reset_index(drop=True)
df['label'] = (df['GENRE'] == 'M').astype(int)

sp = pd.read_csv('split_reference.csv')
emp = df['full_name'].map(lambda s: hashlib.sha1(s.encode()).hexdigest()[:10]).values
assert len(sp) == len(df) and (sp['empreinte'].values == emp).all(), "split_reference.csv ne correspond pas"
idx_tr, idx_va, idx_te = [df.index[sp['ensemble'].values == e] for e in ('train', 'val', 'test')]
y_va, y_te = df.loc[idx_va, 'label'].values, df.loc[idx_te, 'label'].values
print(f"Train={len(idx_tr):,} | Val={len(idx_va):,} | Test={len(idx_te):,}")

lr = make_pipeline(make_union(TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 6), sublinear_tf=True),
                              TfidfVectorizer(analyzer='word', token_pattern=r'\S+')),
                   LogisticRegression(C=2, max_iter=3000))
lr.fit(df.loc[idx_tr, 'full_name'], df.loc[idx_tr, 'label'])
P = {'regression_logistique': dict(val=[lr.predict_proba(df.loc[idx_va, 'full_name'])[:, 1]],
                                   test=[lr.predict_proba(df.loc[idx_te, 'full_name'])[:, 1]])}

## 2. Prédictions des autres modèles

Les réseaux ont une prédiction par graine. Le meilleur gradient boosting est choisi sur la
**validation** parmi ceux enregistrés par `boosting.ipynb`.

In [ ]:
for arch in ('v5', 'v4'):
    files = sorted(glob.glob(os.path.join(FINAL_DIR, f'{arch}_seed*.npz')))
    if files:
        z = [np.load(f) for f in files]
        P[arch] = dict(val=[x['val'] for x in z], test=[x['test'] for x in z])
        print(f"{arch} : {len(files)} graines")

boost = {os.path.basename(f)[:-4]: np.load(f) for f in glob.glob(os.path.join(FINAL_DIR, '*.npz'))
         if not re.match(r'v[45]_seed', os.path.basename(f))}
if boost:
    val_acc = {k: accuracy_score(y_va, v['val'] > 0.5) for k, v in boost.items()}
    best_boost = max(val_acc, key=val_acc.get)
    print("boosting (val acc) :", {k: round(v, 4) for k, v in val_acc.items()}, "-> retenu :", best_boost)
    P[best_boost] = dict(val=[boost[best_boost]['val']], test=[boost[best_boost]['test']])

# Ensembles : moyenne avec la régression logistique (probabilités moyennées sur les graines)
mean = lambda ps: np.mean(ps, axis=0)
for name in [k for k in list(P) if k != 'regression_logistique']:
    P[f'ensemble_lr+{name}'] = {s: [(mean(P[name][s]) + P['regression_logistique'][s][0]) / 2]
                                for s in ('val', 'test')}
print("Modèles évalués :", list(P))

## 3. Résultats sur le test

- Réseaux : moyenne ± écart-type sur les graines ; McNemar et intervalle de confiance calculés
  sur la probabilité moyenne des graines.
- Intervalle de confiance à 95 % de l'accuracy par bootstrap (rééchantillonnage du test).

In [ ]:
def metrics(y, p):
    yp = (p > 0.5).astype(int)
    return dict(acc=accuracy_score(y, yp), f1=f1_score(y, yp, average='macro'), auc=roc_auc_score(y, p))


def mcnemar(p_a, p_b, y):
    e_a, e_b = (p_a > 0.5) != y, (p_b > 0.5) != y
    b, c = int((e_a & ~e_b).sum()), int((~e_a & e_b).sum())
    return b, c, (binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0)


rng = np.random.default_rng(42)
boot_idx = rng.integers(0, len(y_te), (N_BOOT, len(y_te)))
p_lr = P['regression_logistique']['test'][0]
rows = []
for name, d in P.items():
    per_seed = pd.DataFrame([metrics(y_te, p) for p in d['test']])
    p = mean(d['test'])
    ok = ((p > 0.5) == y_te).astype(float)
    lo, hi = np.percentile(ok[boot_idx].mean(axis=1), [2.5, 97.5])
    b, c, pval = mcnemar(p, p_lr, y_te) if name != 'regression_logistique' else (0, 0, 1.0)
    conf = np.maximum(p, 1 - p); keep = conf >= SEUIL_INCERTAIN
    rows.append(dict(modele=name, graines=len(d['test']),
                     acc=per_seed['acc'].mean(), acc_std=per_seed['acc'].std(ddof=0),
                     f1=per_seed['f1'].mean(), auc=per_seed['auc'].mean(),
                     acc_ic95=f"[{lo:.4f} ; {hi:.4f}]",
                     ecart_vs_lr=per_seed['acc'].mean() - metrics(y_te, p_lr)['acc'],
                     faux_seul=b, juste_seul=c, mcnemar_p=pval,
                     taux_reponse=keep.mean(), precision_si_reponse=((p > 0.5) == y_te)[keep].mean(),
                     val_acc=np.mean([accuracy_score(y_va, v > 0.5) for v in d['val']])))
res = pd.DataFrame(rows).sort_values('acc', ascending=False).reset_index(drop=True)
display(res.round(4))

## 4. Choix du modèle de production (règle fixée à l'avance)

In [ ]:
lr_acc = res.loc[res.modele == 'regression_logistique', 'acc'].iloc[0]
meilleurs = res[(res.modele != 'regression_logistique') & (res.acc > lr_acc) & (res.mcnemar_p < 0.05)]
choix = meilleurs.iloc[0]['modele'] if len(meilleurs) else 'regression_logistique'
raison = ("significativement meilleur que la régression logistique (McNemar p < 0,05)" if len(meilleurs)
          else "aucun modèle n'est significativement meilleur : on garde le plus simple")
print(f"Modèle de production : {choix} -> {raison}")
r = res.set_index('modele').loc[choix]
print(f"  test acc = {r['acc']:.4f} {r['acc_ic95']} | F1 = {r['f1']:.4f} | AUC = {r['auc']:.4f}")
print(f"  mode incertain (seuil {SEUIL_INCERTAIN}) : {r['taux_reponse']*100:.1f} % des noms traités, "
      f"précision {r['precision_si_reponse']*100:.2f} %")

## 5. Figure et sauvegarde

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.45 * len(res) + 1.5))
order = res.sort_values('acc')
ax.barh(order['modele'], order['acc'], xerr=order['acc_std'], color=['#E15759' if m == choix else '#4E79A7'
                                                                      for m in order['modele']])
for y, (a, m) in enumerate(zip(order['acc'], order['modele'])):
    ax.text(a + 0.001, y, f"{a:.4f}", va='center', fontsize=9)
ax.set_xlim(order['acc'].min() - 0.02, min(1, order['acc'].max() + 0.015))
ax.set_xlabel('Accuracy sur le test'); ax.set_title('Évaluation finale (test)')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/evaluation_finale.png', dpi=150); plt.show()

with open(os.path.join(FINAL_DIR, 'evaluation_finale.json'), 'w') as f:
    json.dump(dict(choix=choix, raison=raison, seuil_incertain=SEUIL_INCERTAIN,
                   resultats=res.round(4).to_dict('records')), f, indent=1, ensure_ascii=False)
res.round(4).to_csv(os.path.join(FINAL_DIR, 'evaluation_finale.csv'), index=False)
print(f"Résultats enregistrés dans {FINAL_DIR}/")